In [1]:
import subprocess, time, json, os

def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result

sh("kubectl get nodes -o wide")
sh("kubectl get nodes -o custom-columns=NAME:.metadata.name,CPU:.status.allocatable.cpu")
sh("kubectl describe nodes | grep -A 4 'Allocated resources'");

NAME           STATUS   ROLES           AGE   VERSION   INTERNAL-IP    EXTERNAL-IP   OS-IMAGE                         KERNEL-VERSION             CONTAINER-RUNTIME
minikube       Ready    control-plane   28m   v1.37.0   192.168.49.2   <none>        Debian GNU/Linux 12 (bookworm)   7.0.0-30-generic (arm64)   containerd://2.3.4
minikube-m02   Ready    <none>          27m   v1.37.0   192.168.49.3   <none>        Debian GNU/Linux 12 (bookworm)   7.0.0-30-generic (arm64)   containerd://2.3.4

NAME           CPU
minikube       2
minikube-m02   2

Allocated resources:
  (Total limits may be over 100 percent, i.e., overcommitted.)
  Resource           Requests    Limits
  --------           --------    ------
  cpu                850m (42%)  100m (5%)
--
Allocated resources:
  (Total limits may be over 100 percent, i.e., overcommitted.)
  Resource           Requests   Limits
  --------           --------   ------
  cpu                100m (5%)  100m (5%)



In [2]:
%%writefile generate_shards.py
# Generates 8 shards of seeded synthetic user signup records with known number of invalid rows.
import argparse, csv, json, os, random

REQUIRED = ["user_id", "name", "email", "signup_date"]
FIELDS = REQUIRED + ["country"]
NAMES = ["Priya", "Arjun", "Meera", "Rahul", "Ananya", "Vikram", "Sneha", "Karthik"]
DOMAINS = ["gmail.com", "yahoo.com", "outlook.com", "iitm.ac.in"]
COUNTRIES = ["IN", "US", "AU", "SG", "UK"]
BAD_EMAILS = ["{u}.gmail.com", "{u}@", "@gmail.com", "{u}@@gmail.com", "{u} @gmail.com", "{u}@gmail"]


def generate_shard(shard, rows, out_dir):
    rng = random.Random(42 + shard)
    n_bad_email = rng.randint(50, 150)
    n_missing = rng.randint(50, 150)
    picked = rng.sample(range(rows), n_bad_email + n_missing)
    bad_email_rows = set(picked[:n_bad_email])
    missing_rows = set(picked[n_bad_email:])

    with open(f"{out_dir}/shard_{shard}.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        writer.writeheader()
        for i in range(rows):
            user_id = shard * rows + i
            name = rng.choice(NAMES)
            row = {
                "user_id": str(user_id),
                "name": name,
                "email": f"{name.lower()}{user_id}@{rng.choice(DOMAINS)}",
                "signup_date": f"2026-{rng.randint(1, 12):02d}-{rng.randint(1, 28):02d}",
                "country": rng.choice(COUNTRIES),
            }
            if i in bad_email_rows:
                row["email"] = rng.choice(BAD_EMAILS).format(u=f"{name.lower()}{user_id}")
            elif i in missing_rows:
                row[rng.choice(REQUIRED)] = ""
            writer.writerow(row)

    return {"shard": shard, "rows": rows, "bad_email": n_bad_email,
            "missing_field": n_missing, "invalid": n_bad_email + n_missing}


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--out-dir", default="shards")
    parser.add_argument("--n-shards", type=int, default=8)
    parser.add_argument("--rows", type=int, default=200_000)
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    expected = [generate_shard(s, args.rows, args.out_dir) for s in range(args.n_shards)]
    with open(f"{args.out_dir}/expected_counts.json", "w") as f:
        json.dump(expected, f, indent=2)
    for e in expected:
        print(e)

Writing generate_shards.py


In [6]:
subprocess.run(["python3", "generate_shards.py", "--rows", "1000000"], check=True)
sh("ls -lh shards");

{'shard': 0, 'rows': 1000000, 'bad_email': 131, 'missing_field': 64, 'invalid': 195}
{'shard': 1, 'rows': 1000000, 'bad_email': 54, 'missing_field': 86, 'invalid': 140}
{'shard': 2, 'rows': 1000000, 'bad_email': 102, 'missing_field': 116, 'invalid': 218}
{'shard': 3, 'rows': 1000000, 'bad_email': 84, 'missing_field': 103, 'invalid': 187}
{'shard': 4, 'rows': 1000000, 'bad_email': 59, 'missing_field': 101, 'invalid': 160}
{'shard': 5, 'rows': 1000000, 'bad_email': 95, 'missing_field': 58, 'invalid': 153}
{'shard': 6, 'rows': 1000000, 'bad_email': 120, 'missing_field': 90, 'invalid': 210}
{'shard': 7, 'rows': 1000000, 'bad_email': 58, 'missing_field': 94, 'invalid': 152}
total 408M
-rw-rw-r-- 1 rohan rohan 896 Sep 19 03:28 expected_counts.json
-rw-rw-r-- 1 rohan rohan 50M Sep 19 03:28 shard_0.csv
-rw-rw-r-- 1 rohan rohan 52M Sep 19 03:28 shard_1.csv
-rw-rw-r-- 1 rohan rohan 52M Sep 19 03:28 shard_2.csv
-rw-rw-r-- 1 rohan rohan 52M Sep 19 03:28 shard_3.csv
-rw-rw-r-- 1 rohan rohan 52M Sep

In [8]:
%%writefile validate_shard.py
# Entry point for each pod of the Job.
import csv, json, os, re, socket, time

EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
REQUIRED = ["user_id", "name", "email", "signup_date"]


def main():
    shard = int(os.environ.get("JOB_COMPLETION_INDEX", "0"))
    data_dir = os.environ.get("DATA_DIR", "/app/shards")
    pod_name = os.environ.get("POD_NAME", socket.gethostname())
    node_name = os.environ.get("NODE_NAME", "unknown")
    path = f"{data_dir}/shard_{shard}.csv"

    print(f"[shard {shard}] pod={pod_name} node={node_name} file={path}", flush=True)

    t0 = time.time()
    total = missing = bad_email = 0
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            total += 1
            if any(not row[col].strip() for col in REQUIRED):
                missing += 1
            elif not EMAIL_RE.match(row["email"]):
                bad_email += 1

    result = {
        "shard": shard, "rows": total, "missing_field": missing, "bad_email": bad_email,
        "invalid": missing + bad_email, "seconds": round(time.time() - t0, 2),
        "pod_name": pod_name, "node_name": node_name,
    }
    print("RESULT_JSON:" + json.dumps(result), flush=True)


if __name__ == "__main__":
    main()

Overwriting validate_shard.py


In [9]:
expected = json.load(open("shards/expected_counts.json"))
env = os.environ.copy()
env.update({"DATA_DIR": "shards", "POD_NAME": "local-test", "NODE_NAME": "local"})

for idx in [0, 7]:
    env["JOB_COMPLETION_INDEX"] = str(idx)
    out = subprocess.run(["python3", "validate_shard.py"], env=env, capture_output=True, text=True).stdout
    print(out)
    print("expected:", expected[idx], "\n")

[shard 0] pod=local-test node=local file=shards/shard_0.csv
RESULT_JSON:{"shard": 0, "rows": 1000000, "missing_field": 64, "bad_email": 131, "invalid": 195, "seconds": 1.86, "pod_name": "local-test", "node_name": "local"}

expected: {'shard': 0, 'rows': 1000000, 'bad_email': 131, 'missing_field': 64, 'invalid': 195} 

[shard 7] pod=local-test node=local file=shards/shard_7.csv
RESULT_JSON:{"shard": 7, "rows": 1000000, "missing_field": 94, "bad_email": 58, "invalid": 152, "seconds": 1.85, "pod_name": "local-test", "node_name": "local"}

expected: {'shard': 7, 'rows': 1000000, 'bad_email': 58, 'missing_field': 94, 'invalid': 152} 



In [10]:
%%writefile Dockerfile.validator
FROM python:3.11-slim
WORKDIR /app
COPY validate_shard.py .
COPY shards/ ./shards/
CMD ["python", "validate_shard.py"]

Writing Dockerfile.validator


In [11]:
!docker build -t shard-validator:latest -f Dockerfile.validator .


[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (1/2)                                          docker:default
 => [internal] load build definition from Dockerfile.validator             0.0s
 => => transferring dockerfile: 166B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.2s
[+] Building 0.4s (1/2)                                          docker:default
 => [internal] load build definition from Dockerfile.validator             0.0s
 => => transferring dockerfile: 166B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.3s
[+] Building 0.5s (1/2)                                          docker:default
 => [internal] load build definition from Dockerfile.validator             0.0s
 => => transferring dockerfile: 166B                                       0.0s
 => [internal] load metadata for docker

In [12]:
!minikube image load shard-validator:latest

In [13]:
sh("minikube ssh --node minikube \"sudo crictl images | grep shard-validator\"")
sh("minikube ssh --node minikube-m02 \"sudo crictl images | grep shard-validator\"");

docker.io/library/shard-validator         latest               ee194fd73f39f       118MB

docker.io/library/shard-validator         latest               ee194fd73f39f       118MB



In [14]:
%%writefile job-validate.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: shard-validate-job
  labels:
    app: shard-validate
spec:
  completions: 8
  parallelism: 4
  completionMode: Indexed
  backoffLimit: 4
  activeDeadlineSeconds: 600
  template:
    metadata:
      labels:
        app: shard-validate
    spec:
      restartPolicy: Never
      topologySpreadConstraints:
      - maxSkew: 1
        topologyKey: kubernetes.io/hostname
        whenUnsatisfiable: ScheduleAnyway
        labelSelector:
          matchLabels:
            app: shard-validate
      containers:
      - name: validator
        image: shard-validator:latest
        imagePullPolicy: IfNotPresent
        env:
        - name: DATA_DIR
          value: "/app/shards"
        - name: POD_NAME
          valueFrom:
            fieldRef:
              fieldPath: metadata.name
        - name: NODE_NAME
          valueFrom:
            fieldRef:
              fieldPath: spec.nodeName
        resources:
          requests:
            cpu: "500m"
            memory: "128Mi"
          limits:
            cpu: "500m"
            memory: "128Mi"

Writing job-validate.yaml


In [15]:
JOB = "shard-validate-job"
sh(f"kubectl delete job {JOB} --ignore-not-found --wait=true")
sh("kubectl apply -f job-validate.yaml")

best_snapshot, max_running = "", 0
start = time.time()
while time.time() - start < 180:
    snapshot = subprocess.run(f"kubectl get pods -l job-name={JOB} -o wide",
                              shell=True, capture_output=True, text=True).stdout
    running = sum(" Running " in line for line in snapshot.splitlines())
    if running > max_running:
        max_running, best_snapshot = running, snapshot
    succeeded = subprocess.run(f"kubectl get job {JOB} -o jsonpath={{.status.succeeded}}",
                               shell=True, capture_output=True, text=True).stdout
    if succeeded == "8":
        break
    time.sleep(0.5)

print(f"job finished in {time.time() - start:.1f} s")
print(f"max pods running at the same time: {max_running}\n")
print(best_snapshot)


job.batch/shard-validate-job created

job finished in 16.5 s
max pods running at the same time: 4

NAME                         READY   STATUS    RESTARTS   AGE   IP           NODE           NOMINATED NODE   READINESS GATES
shard-validate-job-0-fx7dl   1/1     Running   0          1s    10.244.1.2   minikube-m02   <none>           <none>
shard-validate-job-1-6jqw9   1/1     Running   0          1s    10.244.0.3   minikube       <none>           <none>
shard-validate-job-2-c5t2c   1/1     Running   0          1s    10.244.1.3   minikube-m02   <none>           <none>
shard-validate-job-3-5znwf   1/1     Running   0          1s    10.244.0.4   minikube       <none>           <none>



In [16]:
sh(f"kubectl get pods -l job-name={JOB} -o wide")
sh(f"kubectl get job {JOB}");

NAME                         READY   STATUS      RESTARTS   AGE   IP           NODE           NOMINATED NODE   READINESS GATES
shard-validate-job-0-fx7dl   0/1     Completed   0          47s   10.244.1.2   minikube-m02   <none>           <none>
shard-validate-job-1-6jqw9   0/1     Completed   0          47s   10.244.0.3   minikube       <none>           <none>
shard-validate-job-2-c5t2c   0/1     Completed   0          47s   10.244.1.3   minikube-m02   <none>           <none>
shard-validate-job-3-5znwf   0/1     Completed   0          47s   10.244.0.4   minikube       <none>           <none>
shard-validate-job-4-52md8   0/1     Completed   0          39s   10.244.1.5   minikube-m02   <none>           <none>
shard-validate-job-5-4mnfh   0/1     Completed   0          39s   10.244.1.4   minikube-m02   <none>           <none>
shard-validate-job-6-hbksd   0/1     Completed   0          39s   10.244.0.5   minikube       <none>           <none>
shard-validate-job-7-2vp8b   0/1     Completed 

In [17]:
%%writefile collect_results.py
# Collecting RESULT_JSON lines from every pod via Kubernetes API.
import argparse, json, re, sys
import pandas as pd
from kubernetes import client, config

RESULT_LINE_RE = re.compile(r"RESULT_JSON:(\{.*\})")


def load_kube_config():
    try:
        config.load_kube_config()
    except Exception:
        config.load_incluster_config()


def collect(job_name, namespace="default"):
    load_kube_config()
    v1 = client.CoreV1Api()
    pods = v1.list_namespaced_pod(namespace=namespace, label_selector=f"job-name={job_name}")
    if not pods.items:
        print(f"No pods found for job '{job_name}'.", file=sys.stderr)
        return pd.DataFrame()

    rows = []
    for pod in pods.items:
        pod_name = pod.metadata.name
        try:
            logs = v1.read_namespaced_pod_log(name=pod_name, namespace=namespace)
        except client.exceptions.ApiException as e:
            print(f"  Could not read logs for {pod_name}: {e.reason}", file=sys.stderr)
            continue
        match = RESULT_LINE_RE.search(logs)
        if not match:
            print(f"  No RESULT_JSON line in {pod_name} yet.", file=sys.stderr)
            continue
        result = json.loads(match.group(1))
        result["k8s_pod_phase"] = pod.status.phase
        rows.append(result)

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("shard").reset_index(drop=True)
    return df


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--job-name", required=True)
    parser.add_argument("--namespace", default="default")
    parser.add_argument("--out", default=None)
    args = parser.parse_args()

    df = collect(args.job_name, args.namespace)
    if df.empty:
        print("No results collected.")
    else:
        pd.set_option("display.width", 120)
        print(df.to_string(index=False))
        if args.out:
            df.to_csv(args.out, index=False)

Writing collect_results.py


In [18]:
from collect_results import collect

results_df = collect(JOB)
results_df

,shard,rows,missing_field,bad_email,invalid,seconds,pod_name,node_name,k8s_pod_phase
0,0,1000000,64,131,195,4.42,shard-validate-job-0-fx7dl,minikube-m02,Succeeded
1,1,1000000,86,54,140,4.47,shard-validate-job-1-6jqw9,minikube,Succeeded
2,2,1000000,116,102,218,4.47,shard-validate-job-2-c5t2c,minikube-m02,Succeeded
3,3,1000000,103,84,187,4.38,shard-validate-job-3-5znwf,minikube,Succeeded
4,4,1000000,101,59,160,4.29,shard-validate-job-4-52md8,minikube-m02,Succeeded
5,5,1000000,58,95,153,4.26,shard-validate-job-5-4mnfh,minikube-m02,Succeeded
6,6,1000000,90,120,210,4.19,shard-validate-job-6-hbksd,minikube,Succeeded
7,7,1000000,94,58,152,4.21,shard-validate-job-7-2vp8b,minikube,Succeeded


In [20]:
import pandas as pd

expected_df = pd.DataFrame(json.load(open("shards/expected_counts.json")))
check = results_df[["shard", "invalid", "node_name"]].merge(
    expected_df[["shard", "invalid"]], on="shard", suffixes=("_found", "_expected"))
check["match"] = check["invalid_found"] == check["invalid_expected"]
print(check.to_string(index=False))
print(f"\ntotal invalid rows found: {results_df['invalid'].sum()}")
print(f"all shards match: {check['match'].all()}")

 shard  invalid_found    node_name  invalid_expected  match
     0            195 minikube-m02               195   True
     1            140     minikube               140   True
     2            218 minikube-m02               218   True
     3            187     minikube               187   True
     4            160 minikube-m02               160   True
     5            153 minikube-m02               153   True
     6            210     minikube               210   True
     7            152     minikube               152   True

total invalid rows found: 1415
all shards match: True


In [21]:
sh(f"kubectl delete job {JOB}");

job.batch "shard-validate-job" deleted from default namespace

